# EngSVG multi-family IR-action LoRA
Trains on 8,000 source-group-disjoint examples and evaluates the base and trained model on 100 untouched truss and mechanical-plate edits.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile, torch

def run(args, **kwargs):
    print('RUN:', ' '.join(map(str, args)), flush=True)
    subprocess.run(args, check=True, **kwargs)

assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
bundle = pathlib.Path('/content/engsvg-multifamily-colab-bundle.zip')
assert bundle.exists(), f'Upload {bundle.name} into /content first'
payload = pathlib.Path('/content/engsvg-multifamily-payload')
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(payload)
root = pathlib.Path('/content/SVG')
if not root.exists():
    run(['git', 'clone', '--depth', '1', '--branch', 'cvpr2027-research-package', 'https://github.com/ayushdebnath012/SVG.git', str(root)])
run([sys.executable, '-m', 'pip', 'install', '-q', 'svgpathtools==1.7.2', 'peft', 'transformers', 'accelerate'])
run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
scripts = root / 'cvpr2027/scripts'
for source in (payload / 'scripts').glob('*.py'):
    shutil.copy2(source, scripts / source.name)
env = os.environ.copy()
env['PYTHONPATH'] = f'{scripts}:{root}'
run([sys.executable, str(scripts / 'engsvg_multifamily_train.py'), 'train',
     '--data', str(payload / 'data/engsvg-multifamily-train-v1'),
     '--out', '/content/engsvg-multifamily-run', '--epochs', '1'], env=env)
print('RESULT: /content/engsvg-multifamily-run-artifacts.zip', flush=True)
